# Generate `survey_history.csv` from Beiwe survey history JSON

This notebook creates a more readable report from a Beiwe `*_surveys_history_data.json` file, which can be downloaded using the "Export Survey Edits History JSON" button on Beiwe. 

It will produce a CSV with one row per change event (added/removed/modified question fields) across survey versions.

## Step 1: Set Input and Output Paths

Replace the values below with your file paths before running the rest of the notebook.

In [ ]:
# Replace these paths as needed
history_path = "mypath/***_surveys_history_data.json"
output_csv = "mypath/survey_history.csv"
include_no_change = True  # Set to False if you want to exclude no_change rows


## Step 2: Import Packages

Run this cell to load required Python libraries.

In [2]:
import json
import os
import pandas as pd


## Step 3: Define Helper Functions

These helper functions format values and map survey questions by `question_id`.

In [3]:
def _to_str(v):
    if isinstance(v, (dict, list)):
        return json.dumps(v, sort_keys=True, ensure_ascii=False)
    return "" if v is None else str(v)


def _question_map(survey_json):
    qmap = {}
    for q in (survey_json or []):
        qid = q.get("question_id", "<no_question_id>")
        qmap[qid] = q
    return qmap


## Step 4: Build Survey Version Diff Rows

This cell reads the survey history JSON and computes baseline/added/removed/modified/no-change rows across versions.

In [4]:
with open(history_path, "r") as f:
    history = json.load(f)

rows = []

for survey_id, versions in history.items():
    versions = sorted(versions, key=lambda v: v.get("archive_start", ""))

    prev_map = None
    for i, v in enumerate(versions, start=1):
        ts = v.get("archive_start", "")
        cur_map = _question_map(v.get("survey_json", []))

        if prev_map is None:
            rows.append({
                "survey_id": survey_id,
                "version_index": i,
                "archive_start": ts,
                "change_type": "baseline",
                "question_id": "",
                "field": "",
                "old_value": "",
                "new_value": "",
            })
            prev_map = cur_map
            continue

        prev_ids = set(prev_map.keys())
        cur_ids = set(cur_map.keys())

        added = sorted(cur_ids - prev_ids)
        removed = sorted(prev_ids - cur_ids)
        common = sorted(cur_ids & prev_ids)

        # Added questions
        for qid in added:
            rows.append({
                "survey_id": survey_id,
                "version_index": i,
                "archive_start": ts,
                "change_type": "added_question",
                "question_id": qid,
                "field": "__question__",
                "old_value": "",
                "new_value": _to_str(cur_map[qid]),
            })

        # Removed questions
        for qid in removed:
            rows.append({
                "survey_id": survey_id,
                "version_index": i,
                "archive_start": ts,
                "change_type": "removed_question",
                "question_id": qid,
                "field": "__question__",
                "old_value": _to_str(prev_map[qid]),
                "new_value": "",
            })

        # Modified question fields
        modified_any = False
        for qid in common:
            before = prev_map[qid]
            after = cur_map[qid]
            if before != after:
                modified_any = True
                fields = sorted(set(before.keys()) | set(after.keys()))
                for field in fields:
                    b = before.get(field, None)
                    a = after.get(field, None)
                    if b != a:
                        rows.append({
                            "survey_id": survey_id,
                            "version_index": i,
                            "archive_start": ts,
                            "change_type": "modified_field",
                            "question_id": qid,
                            "field": field,
                            "old_value": _to_str(b),
                            "new_value": _to_str(a),
                        })

        if include_no_change and (not added and not removed and not modified_any):
            rows.append({
                "survey_id": survey_id,
                "version_index": i,
                "archive_start": ts,
                "change_type": "no_change",
                "question_id": "",
                "field": "",
                "old_value": "",
                "new_value": "",
            })

        prev_map = cur_map


## Step 5: Save CSV and Preview Results

This cell writes `survey_history.csv`, prints the output path, and shows a preview.

In [5]:
df_hist = pd.DataFrame(rows)

# Keep a stable column order even if no rows are generated
columns = [
    "survey_id",
    "version_index",
    "archive_start",
    "change_type",
    "question_id",
    "field",
    "old_value",
    "new_value",
]

if df_hist.empty:
    df_hist = pd.DataFrame(columns=columns)
else:
    df_hist = df_hist[columns]
    df_hist.sort_values(["survey_id", "version_index", "change_type", "question_id", "field"], inplace=True)

df_hist.to_csv(output_csv, index=False)

print(f"Output path: {os.path.abspath(output_csv)}")
display(df_hist.head(20))


Output path: /Users/stellenshun.li/Desktop/Beiwe/beiwe/code/forest_mano/survey_history.csv


,survey_id,version_index,archive_start,change_type,question_id,field,old_value,new_value
0,oPZALRMPVPqT6nSfQMkUGf6U,1,2025-12-12T21:36:35.255285+00:00,baseline,,,,
1,oPZALRMPVPqT6nSfQMkUGf6U,2,2025-12-12T21:38:24.875892+00:00,added_question,f7ebe0de-2eab-4a9f-fbd0-e68865a6d20b,__question__,,"{""display_if"": null, ""question_id"": ""f7ebe0de-..."
2,oPZALRMPVPqT6nSfQMkUGf6U,3,2025-12-12T21:38:42.196678+00:00,no_change,,,,
3,oPZALRMPVPqT6nSfQMkUGf6U,4,2025-12-12T21:40:38.240270+00:00,added_question,7a3e9cde-94ad-4b7f-e231-c56229f44dee,__question__,,"{""display_if"": null, ""max"": ""10"", ""min"": ""1"", ..."
4,oPZALRMPVPqT6nSfQMkUGf6U,5,2025-12-12T21:42:53.020161+00:00,added_question,bb83ca6c-b569-4c41-ce4c-1bcac5d1e343,__question__,,"{""answers"": [{""text"": ""No""}, {""text"": ""Yes!""},..."
5,oPZALRMPVPqT6nSfQMkUGf6U,6,2025-12-12T21:47:12.287314+00:00,added_question,165393ea-b923-40b2-c519-1342a2e41e51,__question__,,"{""answers"": [{""text"": ""Pop""}, {""text"": ""Rap""},..."
6,oPZALRMPVPqT6nSfQMkUGf6U,7,2025-12-12T21:47:53.107237+00:00,no_change,,,,
7,oPZALRMPVPqT6nSfQMkUGf6U,8,2025-12-12T21:49:21.645249+00:00,added_question,10912dd2-5aaa-4461-945f-31a06a28fbee,__question__,,"{""display_if"": null, ""question_id"": ""10912dd2-..."
8,oPZALRMPVPqT6nSfQMkUGf6U,8,2025-12-12T21:49:21.645249+00:00,added_question,cd885ed4-fe4e-423d-d0c2-3e86e4bc77cb,__question__,,"{""display_if"": null, ""question_id"": ""cd885ed4-..."
9,oPZALRMPVPqT6nSfQMkUGf6U,8,2025-12-12T21:49:21.645249+00:00,added_question,e2ac37a6-0e2d-4785-f065-3e79a30094a8,__question__,,"{""display_if"": null, ""question_id"": ""e2ac37a6-..."


## Step 6 (Optional): Quick Filtering Examples

Use these examples to quickly inspect changed versions.

In [6]:
# Count each change type
df_hist["change_type"].value_counts(dropna=False)

# Show only modified fields
# df_hist[df_hist["change_type"] == "modified_field"].head(50)

# Show version rows where questions were removed
# df_hist[df_hist["change_type"] == "removed_question"]


change_type
added_question      7
no_change           4
baseline            1
modified_field      1
removed_question    1
Name: count, dtype: int64